# 06 Learned `q(x)` From Synthetic Labels

This notebook is the learning stage of the thesis story.

Demonstration question:
- If the oracle reliability field `q(x)` is not given directly, can we learn a useful approximation from synthetic labels?
- Which model class is good enough to reproduce the **planner behavior**, not just the raw `q(x)` values?
- How different are probability prediction quality and planner-behavior agreement?

This notebook keeps the mechanism from notebook `04` fixed and changes only how `q(x)` is obtained.


## Learning Setup, Formulas, And Why This Stage Matters

Oracle field:

```math
q(x) \in [0,1]
```

Synthetic supervision:

```math
y_i \sim \mathrm{Bernoulli}(q(x_i))
```

Learned reliability model:

```math
\hat q(x) \approx p(y=1 \mid \phi(x))
```

where `phi(x)` is a feature representation of state or image geometry.

Why this stage matters:
- it tests whether the thesis mechanism survives model approximation,
- it separates the oracle mechanism from learning error,
- it turns the project from a hand-designed example into a learning problem.

This notebook compares four model families:
- logistic regression: the simplest linear probabilistic baseline,
- smoothed grid lookup: local nonparametric spatial averaging,
- small MLP: nonlinear direct learning in feature space,
- exact GP regression proxy on a subset: a small-data probabilistic smoother.

Important modeling note:
- the GP block below is a **regression proxy** on Bernoulli labels, not a full GP classifier with a Bernoulli likelihood and Laplace/EP inference.
- that is acceptable here because the goal is comparative notebook insight, not a production GP classifier.


In [ ]:
from pathlib import Path
import sys
import math

repo_root = Path.cwd()
if not (repo_root / "scripts").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.ndimage import gaussian_filter
from scipy.stats import rankdata

from scripts.state_dependent_observation_helpers import (
    boundary_drop_q,
    covariance_logdet_series,
    covariance_trace_series,
    evaluate_q_field_on_grid,
    grid_states,
    make_action_library,
    make_default_camera,
    make_observation_fn,
    observation_covariance,
    orientation_drop_q,
    process_covariance_from_rho,
    radial_drop_q,
    score_action_library,
    simulate_receding_horizon,
)

plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(7)
camera = make_default_camera()
g = make_observation_fn(camera, obs_mode="uv")


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30.0, 30.0)))


def in_frame(obs):
    u, v = obs
    return (
        np.isfinite(u) and np.isfinite(v)
        and 0.0 <= u < camera.img_width
        and 0.0 <= v < camera.img_height
    )


def boundary_distance_px(obs):
    u, v = obs
    return min(u, v, camera.img_width - 1.0 - u, camera.img_height - 1.0 - v)


def oracle_q(state):
    q_radial = radial_drop_q(
        state,
        center=(0.0, 0.0),
        radius=1.0,
        transition=1.0,
        q_min=0.15,
    )
    q_boundary = boundary_drop_q(
        state,
        camera,
        margin_px=180.0,
        q_min=0.20,
    )
    q_heading = orientation_drop_q(
        state,
        preferred_heading=0.0,
        power=1.3,
        q_min=0.35,
    )
    # Product-style composition means the combined field is low when any component is weak.
    return float(
        np.clip(
            q_radial
            * (0.45 + 0.55 * q_boundary)
            * (0.55 + 0.45 * q_heading),
            0.0,
            1.0,
        )
    )


def sample_visible_states(n_samples, *, q_range=None):
    samples = []
    while len(samples) < n_samples:
        state = np.array(
            [
                rng.uniform(-4.3, 4.3),
                rng.uniform(-4.5, 0.6),
                rng.uniform(-math.pi, math.pi),
            ],
            dtype=float,
        )
        if not in_frame(g(state)):
            continue
        q_value = oracle_q(state)
        if q_range is not None and not (q_range[0] <= q_value <= q_range[1]):
            continue
        samples.append(state)
    return np.asarray(samples, dtype=float)


def feature_state(states):
    states = np.asarray(states, dtype=float)
    return np.column_stack(
        [
            states[:, 0],
            states[:, 1],
            np.sin(states[:, 2]),
            np.cos(states[:, 2]),
        ]
    )


def feature_image_aware(states):
    states = np.asarray(states, dtype=float)
    obs = np.asarray([g(state) for state in states], dtype=float)
    boundary_dist = np.asarray([boundary_distance_px(obs_i) for obs_i in obs], dtype=float)
    return np.column_stack(
        [
            states[:, 0],
            states[:, 1],
            np.sin(states[:, 2]),
            np.cos(states[:, 2]),
            obs[:, 0],
            obs[:, 1],
            boundary_dist,
        ]
    )


class Standardizer:
    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean(axis=0)
        self.scale_ = X.std(axis=0)
        self.scale_[self.scale_ < 1e-8] = 1.0
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self.mean_) / self.scale_

    def fit_transform(self, X):
        return self.fit(X).transform(X)


class LogisticModel:
    def __init__(self, reg=1e-2, maxiter=250):
        self.reg = float(reg)
        self.maxiter = int(maxiter)
        self.scaler = Standardizer()

    def fit(self, X, y):
        Xs = self.scaler.fit_transform(np.asarray(X, dtype=float))
        y = np.asarray(y, dtype=float)
        n_samples, n_features = Xs.shape

        def objective(params):
            w = params[:n_features]
            b = params[n_features]
            probs = sigmoid(Xs @ w + b)
            loss = (
                -np.mean(y * np.log(probs + 1e-9) + (1.0 - y) * np.log(1.0 - probs + 1e-9))
                + 0.5 * self.reg * np.sum(w ** 2)
            )
            grad_z = (probs - y) / n_samples
            grad_w = Xs.T @ grad_z + self.reg * w
            grad_b = np.sum(grad_z)
            return loss, np.concatenate([grad_w, [grad_b]])

        init = np.zeros(n_features + 1, dtype=float)
        result = minimize(
            lambda p: objective(p)[0],
            init,
            jac=lambda p: objective(p)[1],
            method="L-BFGS-B",
            options={"maxiter": self.maxiter},
        )
        self.w_ = result.x[:n_features]
        self.b_ = result.x[n_features]
        self.opt_result_ = result
        return self

    def predict_proba(self, X):
        Xs = self.scaler.transform(np.asarray(X, dtype=float))
        probs = np.clip(sigmoid(Xs @ self.w_ + self.b_), 1e-4, 1.0 - 1e-4)
        return np.column_stack([1.0 - probs, probs])


class SmoothedGridModel:
    def __init__(self, nx=34, ny=34, sigma=1.2):
        self.nx = int(nx)
        self.ny = int(ny)
        self.sigma = float(sigma)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self.xmin_ = float(X[:, 0].min())
        self.xmax_ = float(X[:, 0].max())
        self.ymin_ = float(X[:, 1].min())
        self.ymax_ = float(X[:, 1].max())

        xi = np.clip(
            ((X[:, 0] - self.xmin_) / (self.xmax_ - self.xmin_ + 1e-9) * (self.nx - 1)).astype(int),
            0,
            self.nx - 1,
        )
        yi = np.clip(
            ((X[:, 1] - self.ymin_) / (self.ymax_ - self.ymin_ + 1e-9) * (self.ny - 1)).astype(int),
            0,
            self.ny - 1,
        )

        numerator = np.zeros((self.ny, self.nx), dtype=float)
        denominator = np.zeros((self.ny, self.nx), dtype=float)
        for x_idx, y_idx, label in zip(xi, yi, y):
            numerator[y_idx, x_idx] += label
            denominator[y_idx, x_idx] += 1.0

        numerator = gaussian_filter(numerator, self.sigma, mode="nearest")
        denominator = gaussian_filter(denominator, self.sigma, mode="nearest")
        self.grid_ = np.divide(numerator, np.maximum(denominator, 1e-6))
        self.global_mean_ = float(y.mean())
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        outputs = []
        for x_value, y_value in X:
            if (
                x_value < self.xmin_ or x_value > self.xmax_
                or y_value < self.ymin_ or y_value > self.ymax_
            ):
                outputs.append(self.global_mean_)
                continue

            x_norm = (x_value - self.xmin_) / (self.xmax_ - self.xmin_ + 1e-9) * (self.nx - 1)
            y_norm = (y_value - self.ymin_) / (self.ymax_ - self.ymin_ + 1e-9) * (self.ny - 1)
            ix = min(int(np.floor(x_norm)), self.nx - 2)
            iy = min(int(np.floor(y_norm)), self.ny - 2)
            tx = x_norm - ix
            ty = y_norm - iy

            z00 = self.grid_[iy, ix]
            z10 = self.grid_[iy, ix + 1]
            z01 = self.grid_[iy + 1, ix]
            z11 = self.grid_[iy + 1, ix + 1]
            z0 = (1.0 - tx) * z00 + tx * z10
            z1 = (1.0 - tx) * z01 + tx * z11
            outputs.append((1.0 - ty) * z0 + ty * z1)

        probs = np.clip(np.asarray(outputs, dtype=float), 1e-4, 1.0 - 1e-4)
        return np.column_stack([1.0 - probs, probs])


class MLPBinary:
    def __init__(self, hidden=24, lr=0.04, epochs=500, reg=1e-4, seed=7):
        self.hidden = int(hidden)
        self.lr = float(lr)
        self.epochs = int(epochs)
        self.reg = float(reg)
        self.seed = int(seed)
        self.scaler = Standardizer()

    def fit(self, X, y):
        Xs = self.scaler.fit_transform(np.asarray(X, dtype=float))
        y = np.asarray(y, dtype=float).reshape(-1, 1)
        n_samples, n_features = Xs.shape

        rng_local = np.random.default_rng(self.seed)
        self.W1_ = 0.15 * rng_local.standard_normal((n_features, self.hidden)) / np.sqrt(n_features)
        self.b1_ = np.zeros((1, self.hidden), dtype=float)
        self.W2_ = 0.15 * rng_local.standard_normal((self.hidden, 1)) / np.sqrt(self.hidden)
        self.b2_ = np.zeros((1, 1), dtype=float)

        mW1 = np.zeros_like(self.W1_)
        mb1 = np.zeros_like(self.b1_)
        mW2 = np.zeros_like(self.W2_)
        mb2 = np.zeros_like(self.b2_)
        vW1 = np.zeros_like(self.W1_)
        vb1 = np.zeros_like(self.b1_)
        vW2 = np.zeros_like(self.W2_)
        vb2 = np.zeros_like(self.b2_)

        beta1 = 0.9
        beta2 = 0.999
        eps = 1e-8
        self.loss_history_ = []

        for step in range(1, self.epochs + 1):
            hidden = np.tanh(Xs @ self.W1_ + self.b1_)
            probs = sigmoid(hidden @ self.W2_ + self.b2_)
            loss = (
                -np.mean(y * np.log(probs + 1e-9) + (1.0 - y) * np.log(1.0 - probs + 1e-9))
                + 0.5 * self.reg * (np.sum(self.W1_ ** 2) + np.sum(self.W2_ ** 2))
            )
            self.loss_history_.append(float(loss))

            dZ2 = (probs - y) / n_samples
            gW2 = hidden.T @ dZ2 + self.reg * self.W2_
            gb2 = np.sum(dZ2, axis=0, keepdims=True)
            dHidden = dZ2 @ self.W2_.T
            dZ1 = dHidden * (1.0 - hidden ** 2)
            gW1 = Xs.T @ dZ1 + self.reg * self.W1_
            gb1 = np.sum(dZ1, axis=0, keepdims=True)

            gradients = [(gW1, "W1"), (gb1, "b1"), (gW2, "W2"), (gb2, "b2")]
            for grad, name in gradients:
                if name == "W1":
                    mW1 = beta1 * mW1 + (1.0 - beta1) * grad
                    vW1 = beta2 * vW1 + (1.0 - beta2) * (grad ** 2)
                    update = self.lr * (mW1 / (1.0 - beta1 ** step)) / (np.sqrt(vW1 / (1.0 - beta2 ** step)) + eps)
                    self.W1_ -= update
                elif name == "b1":
                    mb1 = beta1 * mb1 + (1.0 - beta1) * grad
                    vb1 = beta2 * vb1 + (1.0 - beta2) * (grad ** 2)
                    update = self.lr * (mb1 / (1.0 - beta1 ** step)) / (np.sqrt(vb1 / (1.0 - beta2 ** step)) + eps)
                    self.b1_ -= update
                elif name == "W2":
                    mW2 = beta1 * mW2 + (1.0 - beta1) * grad
                    vW2 = beta2 * vW2 + (1.0 - beta2) * (grad ** 2)
                    update = self.lr * (mW2 / (1.0 - beta1 ** step)) / (np.sqrt(vW2 / (1.0 - beta2 ** step)) + eps)
                    self.W2_ -= update
                else:
                    mb2 = beta1 * mb2 + (1.0 - beta1) * grad
                    vb2 = beta2 * vb2 + (1.0 - beta2) * (grad ** 2)
                    update = self.lr * (mb2 / (1.0 - beta1 ** step)) / (np.sqrt(vb2 / (1.0 - beta2 ** step)) + eps)
                    self.b2_ -= update

        return self

    def predict_proba(self, X):
        Xs = self.scaler.transform(np.asarray(X, dtype=float))
        hidden = np.tanh(Xs @ self.W1_ + self.b1_)
        probs = np.clip(sigmoid(hidden @ self.W2_ + self.b2_)[:, 0], 1e-4, 1.0 - 1e-4)
        return np.column_stack([1.0 - probs, probs])


class ExactGPRegressor:
    def __init__(self, lengthscale=1.2, sigma_f=1.0, sigma_n=0.28, max_points=280):
        self.lengthscale = float(lengthscale)
        self.sigma_f = float(sigma_f)
        self.sigma_n = float(sigma_n)
        self.max_points = int(max_points)
        self.scaler = Standardizer()

    def _kernel(self, XA, XB):
        sq_dist = np.sum((XA[:, None, :] - XB[None, :, :]) ** 2, axis=-1)
        return (self.sigma_f ** 2) * np.exp(-0.5 * sq_dist / (self.lengthscale ** 2))

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        if X.shape[0] > self.max_points:
            subset = np.linspace(0, X.shape[0] - 1, self.max_points).astype(int)
            X = X[subset]
            y = y[subset]

        Xs = self.scaler.fit_transform(X)
        K = self._kernel(Xs, Xs) + (self.sigma_n ** 2 + 1e-6) * np.eye(Xs.shape[0])
        self.X_train_ = Xs
        self.y_train_ = y
        self.L_ = np.linalg.cholesky(K)
        self.alpha_ = np.linalg.solve(self.L_.T, np.linalg.solve(self.L_, y))
        return self

    def predict_proba(self, X):
        Xs = self.scaler.transform(np.asarray(X, dtype=float))
        mean = self._kernel(Xs, self.X_train_) @ self.alpha_
        probs = np.clip(mean, 1e-4, 1.0 - 1e-4)
        return np.column_stack([1.0 - probs, probs])


def brier_score(y_true, probs):
    y_true = np.asarray(y_true, dtype=float)
    probs = np.asarray(probs, dtype=float)
    return float(np.mean((probs - y_true) ** 2))


def negative_log_likelihood(y_true, probs):
    y_true = np.asarray(y_true, dtype=float)
    probs = np.asarray(probs, dtype=float)
    return float(-np.mean(y_true * np.log(probs + 1e-9) + (1.0 - y_true) * np.log(1.0 - probs + 1e-9)))


def auc_score(y_true, probs):
    y_true = np.asarray(y_true, dtype=int)
    probs = np.asarray(probs, dtype=float)
    n_pos = int(np.sum(y_true == 1))
    n_neg = int(np.sum(y_true == 0))
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = rankdata(probs)
    return float((np.sum(ranks[y_true == 1]) - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))


def calibration_curve(y_true, probs, n_bins=10):
    y_true = np.asarray(y_true, dtype=float)
    probs = np.asarray(probs, dtype=float)
    edges = np.linspace(0.0, 1.0, int(n_bins) + 1)
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        if hi == 1.0:
            mask = (probs >= lo) & (probs <= hi)
        else:
            mask = (probs >= lo) & (probs < hi)
        if not np.any(mask):
            rows.append((0.5 * (lo + hi), np.nan, np.nan, 0))
            continue
        rows.append(
            (
                0.5 * (lo + hi),
                float(np.mean(probs[mask])),
                float(np.mean(y_true[mask])),
                int(np.sum(mask)),
            )
        )
    return pd.DataFrame(rows, columns=["bin_center", "mean_pred", "empirical_rate", "count"])


def make_model_q_fn(model, feature_fn):
    def q_fn(state):
        features = feature_fn(np.asarray(state, dtype=float)[None, :])
        return float(model.predict_proba(features)[0, 1])
    return q_fn


def predict_on_grid(q_fn, states_grid):
    return evaluate_q_field_on_grid(q_fn, states_grid)


## Synthetic Dataset Design

The dataset is sampled from states that are actually visible to the notebook camera model.

Why that matters:
- it prevents the learner from spending most of its capacity on obviously invalid out-of-frame states,
- it keeps the learning problem aligned with the camera geometry already used by the planner,
- it makes feature sets with image-space terms meaningful.

Feature sets used in this notebook:
- `state`: `[x, y, sin(theta), cos(theta)]`
- `image-aware`: `[x, y, sin(theta), cos(theta), u, v, d_boundary]`

This lets us ask a useful question:
- does adding image-space geometry help a simple linear classifier, or is the state-space nonlinearity the bigger issue?


In [ ]:
n_train = 2500
n_test = 1200

states_train = sample_visible_states(n_train)
states_test = sample_visible_states(n_test)

q_train = np.asarray([oracle_q(state) for state in states_train], dtype=float)
q_test = np.asarray([oracle_q(state) for state in states_test], dtype=float)

y_train = rng.binomial(1, q_train)
y_test = rng.binomial(1, q_test)

print("Train size:", states_train.shape[0], "Test size:", states_test.shape[0])
print("Train mean q(x):", float(q_train.mean()))
print("Test mean q(x): ", float(q_test.mean()))
print("Observed train detection rate:", float(y_train.mean()))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), constrained_layout=True)

scatter = axes[0].scatter(
    states_train[:, 0],
    states_train[:, 1],
    c=q_train,
    s=10,
    cmap="viridis",
    vmin=0.0,
    vmax=1.0,
)
axes[0].set_title("Training states colored by oracle q(x)")
axes[0].set_xlabel("x [m]")
axes[0].set_ylabel("y [m]")
axes[0].axis("equal")
fig.colorbar(scatter, ax=axes[0], shrink=0.82)

axes[1].hist(q_train, bins=24, color="tab:blue", alpha=0.85)
axes[1].set_title("Oracle q(x) distribution in the training set")
axes[1].set_xlabel("oracle q(x)")
axes[1].set_ylabel("count")

axes[2].scatter(q_train, y_train + 0.03 * rng.standard_normal(y_train.shape[0]), s=8, alpha=0.22)
axes[2].set_title("Synthetic Bernoulli labels from oracle q(x)")
axes[2].set_xlabel("oracle q(x)")
axes[2].set_ylabel("sampled detection label")
axes[2].set_ylim(-0.2, 1.2)

plt.show()


## Model Classes And Why They Are Included

Why these specific models:
- logistic regression: the most defensible linear probability baseline,
- smoothed grid lookup: a direct spatial baseline with almost no representation learning,
- small MLP: a lightweight nonlinear learner that can represent interactions between state terms,
- GP regression proxy: a smooth nonparametric small-data model with explicit kernel bias.

What counts as success here:
- low Brier / NLL and good calibration,
- low RMSE to the oracle field,
- high planner-action agreement with the oracle planner on ambiguity-sensitive benchmark states.

What can fail:
- a model can predict `q(x)` well on average but still change planner ranking at the wrong states,
- a model can be smooth and calibrated yet too conservative near sharp low-`q` transitions,
- a model can overfit the Bernoulli labels and lose spatial structure.


In [ ]:
models = {
    "LogReg(state)": {
        "model": LogisticModel(reg=1e-2, maxiter=250),
        "feature_fn": feature_state,
        "note": "Linear probabilistic baseline in state features.",
    },
    "LogReg(image-aware)": {
        "model": LogisticModel(reg=1e-2, maxiter=250),
        "feature_fn": feature_image_aware,
        "note": "Same linear model, but with image geometry features.",
    },
    "Grid(xy)": {
        "model": SmoothedGridModel(nx=34, ny=34, sigma=1.2),
        "feature_fn": lambda S: np.asarray(S, dtype=float)[:, :2],
        "note": "Smoothed empirical detection rate over (x, y).",
    },
    "MLP(state)": {
        "model": MLPBinary(hidden=24, lr=0.04, epochs=500, reg=1e-4, seed=7),
        "feature_fn": feature_state,
        "note": "Nonlinear state-space learner.",
    },
    "GP(state subset)": {
        "model": ExactGPRegressor(lengthscale=1.2, sigma_f=1.0, sigma_n=0.28, max_points=280),
        "feature_fn": feature_state,
        "note": "Exact GP regression proxy on a subset; smooth but expensive.",
    },
}

metric_rows = []
calibration_tables = {}
q_functions = {}

for name, spec in models.items():
    model = spec["model"]
    feature_fn = spec["feature_fn"]
    X_train = feature_fn(states_train)
    X_test = feature_fn(states_test)

    model.fit(X_train, y_train)
    probs_test = model.predict_proba(X_test)[:, 1]
    calibration_tables[name] = calibration_curve(y_test, probs_test, n_bins=10)
    q_functions[name] = make_model_q_fn(model, feature_fn)

    metric_rows.append(
        {
            "model": name,
            "brier": brier_score(y_test, probs_test),
            "nll": negative_log_likelihood(y_test, probs_test),
            "auc": auc_score(y_test, probs_test),
            "rmse_to_oracle": rmse(q_test, probs_test),
            "note": spec["note"],
        }
    )

metrics_df = pd.DataFrame(metric_rows).sort_values(["rmse_to_oracle", "brier"]).reset_index(drop=True)
metrics_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)

for name in ["LogReg(state)", "Grid(xy)", "MLP(state)"]:
    calib = calibration_tables[name]
    axes[0].plot(calib["mean_pred"], calib["empirical_rate"], marker="o", label=name)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)
axes[0].set_title("Calibration curves")
axes[0].set_xlabel("mean predicted probability")
axes[0].set_ylabel("empirical detection rate")
axes[0].legend()

axes[1].bar(metrics_df["model"], metrics_df["rmse_to_oracle"], color="tab:blue")
axes[1].set_title("RMSE to oracle q(x)")
axes[1].set_ylabel("RMSE")
axes[1].tick_params(axis="x", rotation=40)

mlp_model = models["MLP(state)"]["model"]
axes[2].plot(mlp_model.loss_history_, color="tab:purple")
axes[2].set_title("MLP training loss")
axes[2].set_xlabel("epoch")
axes[2].set_ylabel("binary cross-entropy + L2")

plt.show()


In [ ]:
xs, ys, states_grid = grid_states(xmin=-4.5, xmax=4.5, ymin=-4.5, ymax=0.8, nx=101, ny=101, theta=0.0)
extent = [xs.min(), xs.max(), ys.min(), ys.max()]

q_oracle_map = predict_on_grid(oracle_q, states_grid)
q_logreg_map = predict_on_grid(q_functions["LogReg(state)"], states_grid)
q_grid_map = predict_on_grid(q_functions["Grid(xy)"], states_grid)
q_mlp_map = predict_on_grid(q_functions["MLP(state)"], states_grid)

fig, axes = plt.subplots(2, 3, figsize=(14, 9), constrained_layout=True)
heatmaps = [
    (q_oracle_map, "Oracle q(x)"),
    (q_logreg_map, "LogReg(state)"),
    (q_grid_map, "Grid(xy)"),
    (q_mlp_map, "MLP(state)"),
    (np.abs(q_logreg_map - q_oracle_map), "|LogReg - oracle|"),
    (np.abs(q_grid_map - q_oracle_map), "|Grid - oracle|"),
]

for ax, (values, title) in zip(axes.flat, heatmaps):
    im = ax.imshow(values, origin="lower", extent=extent, vmin=0.0, vmax=1.0, cmap="viridis")
    if "oracle" in title.lower() or "grid" in title.lower() or "logreg" in title.lower() or "mlp" in title.lower():
        pass
    if "|" in title:
        im = ax.imshow(values, origin="lower", extent=extent, cmap="magma")
    ax.set_title(title)
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.axis("equal")
    fig.colorbar(im, ax=ax, shrink=0.78)

plt.show()


## Planner-Behavior Agreement: The Metric That Matters Most

Prediction quality is necessary but not sufficient.

For the thesis question, the main test is:
- if the oracle planner would change its behavior because of `q(x)`,
- does the learned model produce the same action choice at the same belief state?

The benchmark below intentionally focuses on **moderate-`q` states**.

Why:
- very high-`q` states are too easy because all models behave almost like constant `R`,
- extremely low-`q` states can make every model conservative,
- the interesting difference is in the transition region where planner ranking is sensitive.


In [ ]:
dt = 0.2
Q = process_covariance_from_rho(dt=dt, rho_xy=1e-2)
R_good = observation_covariance(obs_mode="uv", uv_std=2.5)
R_bad = observation_covariance(obs_mode="uv", uv_std=20.0)
actions = make_action_library(
    v_values=(0.0, 0.22, 0.35, 0.45),
    w_values=(-1.2, -0.8, -0.35, 0.0, 0.35, 0.8, 1.2),
)

goal_state = np.array([0.8, 0.4, 0.0], dtype=float)
goal_obs = g(goal_state)
goal_obs_cov = observation_covariance(obs_mode="uv", uv_std=24.0)
benchmark_cov = np.diag([0.30, 0.30, 0.12])

benchmark_states = sample_visible_states(30, q_range=(0.10, 0.50))
oracle_best_actions = []
for state in benchmark_states:
    best = score_action_library(
        state,
        benchmark_cov,
        actions,
        dt=dt,
        Q=Q,
        g=g,
        goal_obs=goal_obs,
        goal_obs_cov=goal_obs_cov,
        R_good=R_good,
        R_bad=R_bad,
        q_fn=oracle_q,
        horizon=8,
        risk_weight=1.0,
        ambiguity_weight=1.0,
        control_weight=0.1,
        approx="ET2",
        add_ambiguity=True,
    )[0].control
    oracle_best_actions.append(tuple(best))

planner_rows = []
for name in metrics_df["model"]:
    matches = []
    q_fn_model = q_functions[name]
    for state, oracle_action in zip(benchmark_states, oracle_best_actions):
        best = score_action_library(
            state,
            benchmark_cov,
            actions,
            dt=dt,
            Q=Q,
            g=g,
            goal_obs=goal_obs,
            goal_obs_cov=goal_obs_cov,
            R_good=R_good,
            R_bad=R_bad,
            q_fn=q_fn_model,
            horizon=8,
            risk_weight=1.0,
            ambiguity_weight=1.0,
            control_weight=0.1,
            approx="ET2",
            add_ambiguity=True,
        )[0].control
        matches.append(tuple(best) == oracle_action)

    planner_rows.append(
        {
            "model": name,
            "planner_agreement": float(np.mean(matches)),
        }
    )

planner_df = pd.DataFrame(planner_rows)
summary_df = metrics_df.merge(planner_df, on="model").sort_values(
    ["planner_agreement", "rmse_to_oracle"],
    ascending=[False, True],
).reset_index(drop=True)
summary_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)

axes[0].bar(summary_df["model"], summary_df["planner_agreement"], color="tab:orange")
axes[0].set_ylim(0.0, 1.0)
axes[0].set_title("Planner-action agreement with oracle")
axes[0].set_ylabel("fraction of benchmark states with same best action")
axes[0].tick_params(axis="x", rotation=40)

axes[1].scatter(summary_df["rmse_to_oracle"], summary_df["planner_agreement"], s=90)
for _, row in summary_df.iterrows():
    axes[1].annotate(row["model"], (row["rmse_to_oracle"], row["planner_agreement"]), fontsize=8)
axes[1].set_title("Prediction error vs planner agreement")
axes[1].set_xlabel("RMSE to oracle q(x)")
axes[1].set_ylabel("planner agreement")

plt.show()


In [ ]:
# Pick a representative state where the oracle and logistic baseline disagree if possible.
representative_state = np.array([-0.8, -0.4, 0.0], dtype=float)
representative_source = "fixed thesis state from notebook 04"
for state, oracle_action in zip(benchmark_states, oracle_best_actions):
    oracle_tuple = tuple(oracle_action)
    logreg_tuple = tuple(
        score_action_library(
            state,
            benchmark_cov,
            actions,
            dt=dt,
            Q=Q,
            g=g,
            goal_obs=goal_obs,
            goal_obs_cov=goal_obs_cov,
            R_good=R_good,
            R_bad=R_bad,
            q_fn=q_functions["LogReg(state)"],
            horizon=8,
            risk_weight=1.0,
            ambiguity_weight=1.0,
            control_weight=0.1,
            approx="ET2",
            add_ambiguity=True,
        )[0].control
    )
    grid_tuple = tuple(
        score_action_library(
            state,
            benchmark_cov,
            actions,
            dt=dt,
            Q=Q,
            g=g,
            goal_obs=goal_obs,
            goal_obs_cov=goal_obs_cov,
            R_good=R_good,
            R_bad=R_bad,
            q_fn=q_functions["Grid(xy)"],
            horizon=8,
            risk_weight=1.0,
            ambiguity_weight=1.0,
            control_weight=0.1,
            approx="ET2",
            add_ambiguity=True,
        )[0].control
    )
    if oracle_tuple != logreg_tuple and oracle_tuple == grid_tuple:
        representative_state = state.copy()
        representative_source = "benchmark state where oracle and logistic disagree"
        break

print("Representative state source:", representative_source)
print("Representative state:", representative_state)
print("Oracle q(state):", oracle_q(representative_state))

run_specs = {
    "constant R": None,
    "oracle q(x)": oracle_q,
    "Grid(xy)": q_functions["Grid(xy)"],
    "MLP(state)": q_functions["MLP(state)"],
}

runs = {}
first_actions = {}
for label, q_fn in run_specs.items():
    runs[label] = simulate_receding_horizon(
        representative_state,
        benchmark_cov,
        actions,
        dt=dt,
        Q=Q,
        g=g,
        goal_obs=goal_obs,
        goal_obs_cov=goal_obs_cov,
        R_good=R_good,
        R_bad=R_good if q_fn is None else R_bad,
        q_fn=q_fn,
        horizon=8,
        n_steps=16,
        risk_weight=1.0,
        ambiguity_weight=1.0,
        control_weight=0.1,
        approx="ET2",
        add_ambiguity=True,
    )
    first_actions[label] = runs[label]["controls"][0]

pd.DataFrame(
    [
        {
            "model": label,
            "first_action_v": float(action[0]),
            "first_action_w": float(action[1]),
            "mean_q": float(np.mean(run["q_values"])) if run["q_values"].size else 1.0,
            "min_q": float(np.min(run["q_values"])) if run["q_values"].size else 1.0,
            "final_trace": float(np.trace(run["covs"][-1])),
        }
        for label, action, run in [(label, first_actions[label], runs[label]) for label in runs]
    ]
)


In [ ]:
q_map_full = predict_on_grid(oracle_q, states_grid)

fig, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)

axes[0, 0].imshow(q_map_full, origin="lower", extent=extent, vmin=0.0, vmax=1.0, cmap="viridis", alpha=0.88)
for label, run in runs.items():
    axes[0, 0].plot(run["means"][:, 0], run["means"][:, 1], linewidth=2, label=label)
axes[0, 0].scatter(goal_state[0], goal_state[1], marker="*", s=180, color="tab:red", label="goal")
axes[0, 0].scatter(representative_state[0], representative_state[1], marker="o", s=60, color="white", edgecolor="black", label="start")
axes[0, 0].set_title("Representative rollout comparison on oracle q(x)")
axes[0, 0].set_xlabel("x [m]")
axes[0, 0].set_ylabel("y [m]")
axes[0, 0].axis("equal")
axes[0, 0].legend(loc="best")

for label, run in runs.items():
    axes[0, 1].plot(covariance_trace_series(run["covs"]), marker="o", label=label)
axes[0, 1].set_title("Covariance trace over time")
axes[0, 1].set_xlabel("step")
axes[0, 1].set_ylabel("trace(S)")
axes[0, 1].legend()

for label, run in runs.items():
    axes[1, 0].plot(run["ambiguity_terms"], marker="o", label=label)
axes[1, 0].set_title("Ambiguity over time")
axes[1, 0].set_xlabel("step")
axes[1, 0].set_ylabel("ambiguity")
axes[1, 0].legend()

for label, run in runs.items():
    axes[1, 1].plot(run["q_values"], marker="o", label=label)
axes[1, 1].set_title("Encountered q(x) over time")
axes[1, 1].set_xlabel("step")
axes[1, 1].set_ylabel("q(x)")
axes[1, 1].legend()

plt.show()


## Takeaways, Limitations, Alternatives, Sources

Main takeaways this notebook is meant to support:
- learned `q(x)` can reproduce part of the oracle mechanism without having access to the oracle field directly,
- better probability prediction does **not automatically** imply better planner-behavior agreement,
- simple nonparametric models can be surprisingly competitive when the true field is strongly spatial.

Limitations:
- labels are synthetic rather than collected from a real detector,
- the GP block is a regression proxy, not a full GP classifier,
- all results still depend on the notebook camera geometry and the chosen oracle field family.

Natural next steps:
- train on real detection logs instead of synthetic Bernoulli labels,
- compare state-only and image-conditioned features more systematically,
- replace the GP proxy with a proper GP classifier or sparse GP approximation,
- compare learned direct `q(x)` against occupancy-derived visibility in the same benchmark suite.

Local anchors:
- [`03_synthetic_reliability_fields.ipynb`](03_synthetic_reliability_fields.ipynb)
- [`04_efe_with_state_dependent_observation.ipynb`](04_efe_with_state_dependent_observation.ipynb)
- [`05_map_based_visibility_reference.ipynb`](05_map_based_visibility_reference.ipynb)
- [`state_dependent_observation_helpers.py`](state_dependent_observation_helpers.py)
- [`../docs/state_dependent_observation_plan.md`](../docs/state_dependent_observation_plan.md)

Background references:
- Bishop, *Pattern Recognition and Machine Learning* (2006): logistic regression and MLP background.
- Niculescu-Mizil and Caruana, *Predicting Good Probabilities With Supervised Learning* (ICML 2005): calibration perspective.
- Rasmussen and Williams, *Gaussian Processes for Machine Learning* (2006): https://gaussianprocess.org/gpml/
- Friston et al., *Active inference and epistemic value* (2015): https://pubmed.ncbi.nlm.nih.gov/25689102/
- Friston et al., *Active Inference, Curiosity and Insight* (2017): https://pubmed.ncbi.nlm.nih.gov/28777724/
